<a href="https://colab.research.google.com/github/ust-buan-640/buan-640-notebooks/blob/main/notebooks/m10-1-comparing-groups.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparing More Than Two Groups

In Module 9 we compared two groups. In this notebook we compare three, see why running a t-test for every pair is a problem, and meet the F statistic.

Run the cells in order, from top to bottom, with **Shift + Enter**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

## 1. The data

Test scores (out of 10) of students in three sections of the same course: morning, afternoon, and evening, with 30 students in each.

In [ ]:
test_scores = pd.read_csv("https://raw.githubusercontent.com/ust-buan-640/buan-640-notebooks/main/data/test_scores_eng.csv")
scores = test_scores.melt(var_name="section", value_name="score")
scores.groupby("section", sort=False)["score"].agg(["count", "mean", "std"]).round(2)

In [ ]:
plt.figure(figsize=(9, 4))
sns.swarmplot(x="score", y="section", data=scores, size=5, color="gray")
plt.xlabel("Score")
plt.ylabel("")
plt.title("Scores by section")
plt.show()

The sample means differ: 7.92, 7.50, and 7.35. Sample means always differ a little, even when the population means are equal. The question is whether these differences are larger than what chance alone would produce.

## 2. Why not a t-test for every pair?

With three groups there are three pairs: morning vs. afternoon, morning vs. evening, afternoon vs. evening. Each t-test at α = 0.05 has a 5% chance of a false "significant" result when there is no real difference. With several tests on the same data, the chance of at least one false result is larger than 5%, and it grows with the number of groups.

We can see this with a simulation. We create a world where all three population means are equal, draw three samples of 30, run the three t-tests, and record whether at least one of them is "significant". Then we repeat this many times.

In [ ]:
rng = np.random.default_rng(1)
n_sim = 5000
at_least_one = 0

for _ in range(n_sim):
    a, b, c = rng.normal(7.5, 1.35, size=(3, 30))
    p_values = [stats.ttest_ind(a, b).pvalue, stats.ttest_ind(a, c).pvalue, stats.ttest_ind(b, c).pvalue]
    if min(p_values) < 0.05:
        at_least_one += 1

print(f"Share of simulations with at least one 'significant' pair: {at_least_one / n_sim:.3f}")

About 12% of the simulated studies find at least one "significant" difference, even though no difference exists. This is the problem in the comic below: test enough comparisons and something will come out significant by chance.

![xkcd 882](https://imgs.xkcd.com/comics/significant.png)

*Source: [xkcd 882, "Significant"](https://xkcd.com/882/), Randall Munroe, CC BY-NC 2.5.*

The solution is to ask one question about all groups at once: are the group means further apart than we would expect if all population means were equal?

## 3. Between and within

There are two kinds of variation in the data. The group means vary around the overall mean: this is variation **between** groups. Individual scores vary around their own group mean: this is variation **within** groups. If the population means are all equal, the between-group variation is only the echo of the within-group variation. The **F statistic** compares the two.

In [ ]:
groups = [test_scores[c] for c in test_scores.columns]
f_stat, p_value = stats.f_oneway(*groups)
print(f"F = {f_stat:.3f}, p-value = {p_value:.3f}")

## 4. A shuffle test with three groups

In Module 9 we judged a difference between two groups by shuffling the scores between the groups many times and seeing how often the shuffled difference was as large as the observed one. We can do the same with three groups, using F as the measure of how far apart the group means are.

In [ ]:
all_scores = scores["score"].to_numpy()
rng = np.random.default_rng(2)
n_shuffles = 10000
shuffled_f = []

for _ in range(n_shuffles):
    s = rng.permutation(all_scores)
    shuffled_f.append(stats.f_oneway(s[:30], s[30:60], s[60:]).statistic)

shuffled_f = np.array(shuffled_f)
shuffle_p = np.mean(shuffled_f >= f_stat)

plt.hist(shuffled_f, bins=50, edgecolor="black", color="lightgray")
plt.axvline(f_stat, color="black", linestyle="--", label=f"Observed F = {f_stat:.2f}")
plt.xlabel("F statistic in shuffled data")
plt.ylabel("# of shuffles")
plt.legend()
plt.show()

print(f"Share of shuffles with F at least as large as observed: {shuffle_p:.3f}")
print(f"p-value from the F distribution (ANOVA):              {p_value:.3f}")

The two p-values are nearly the same. The F distribution is the shortcut that tells us, without shuffling, how large F tends to be when all population means are equal.

**Question to you:** at α = 0.05, what do you conclude about the mean scores in the three sections? Does this mean that the three population means are equal?

In the next notebook we read the full ANOVA table produced by software.